## Tokenizer from scratch

In [ ]:
import re
corpus='''Gong Ji-hyeok and Go Da-rim work ? together to bring Go Da-rim’s ex-boyfriend into a business deal. But Ji-hyeok, a usually sharp and calm team leader, is caught off guard when he is kissed by Da-rim. Their relationship ends abruptly as Da-rim quickly departs back home as she hears news from her family. Go Da-rim, hiding the fact she is a single woman, pretends to be a mother to earn a living and pay her family’s debt and mother’s medical bills. On her first day she realises Gong Ji-hyeok is her boss. Da-rim needs to continue the pretence as they work together to pay the debt but the initial goals start to give way to unexpected feelings.'''

#create vocab
words=re.split(r"([.,:;?'()\"]|--|\s)",corpus)
vocab=[item.strip() for item in words if item.strip()]
print(vocab)

#vocab annalysis
all_words = sorted(set(vocab))
vocab_size = len(all_words)

print(vocab_size)

#vocab token pairs
vocab = {token:integer for integer,token in enumerate(all_words)}


#tokenizer version 1
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

#version 2 where the tokenizer can identify unknown wordds and end of sentence
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

['Gong', 'Ji-hyeok', 'and', 'Go', 'Da-rim', 'work', '?', 'together', 'to', 'bring', 'Go', 'Da-rim’s', 'ex-boyfriend', 'into', 'a', 'business', 'deal', '.', 'But', 'Ji-hyeok', ',', 'a', 'usually', 'sharp', 'and', 'calm', 'team', 'leader', ',', 'is', 'caught', 'off', 'guard', 'when', 'he', 'is', 'kissed', 'by', 'Da-rim', '.', 'Their', 'relationship', 'ends', 'abruptly', 'as', 'Da-rim', 'quickly', 'departs', 'back', 'home', 'as', 'she', 'hears', 'news', 'from', 'her', 'family', '.', 'Go', 'Da-rim', ',', 'hiding', 'the', 'fact', 'she', 'is', 'a', 'single', 'woman', ',', 'pretends', 'to', 'be', 'a', 'mother', 'to', 'earn', 'a', 'living', 'and', 'pay', 'her', 'family’s', 'debt', 'and', 'mother’s', 'medical', 'bills', '.', 'On', 'her', 'first', 'day', 'she', 'realises', 'Gong', 'Ji-hyeok', 'is', 'her', 'boss', '.', 'Da-rim', 'needs', 'to', 'continue', 'the', 'pretence', 'as', 'they', 'work', 'together', 'to', 'pay', 'the', 'debt', 'but', 'the', 'initial', 'goals', 'start', 'to', 'give', 'way'

## Bit pair encoder



In [ ]:
from collections import defaultdict

def learn_bpe(corpus,num_merges=3):

  vocab=defaultdict(int)

  for sentence in corpus.split("."):
    words=sentence.strip().split()
    for word in words:
      chars = ['<'] + list(word) + ['>']
      for i in range(len(chars) - 1):
        pair = (chars[i], chars[i+1])
        vocab[pair] += 1
      merges = []
      for _ in range(num_merges):
        if not vocab:
            break

        most_frequent = max(vocab, key=lambda x: vocab[x])
        merges.append(most_frequent)


        new_char = ''.join(most_frequent)
        new_vocab = defaultdict(int)
        for pair in vocab:
            count = vocab[pair]
            if pair == most_frequent:
                continue
            new_pair = list(pair)
            if new_pair[0] == most_frequent[0] and new_pair[1] == most_frequent[1]:
                new_pair[0] = new_char
                new_pair.pop(1)
            new_vocab[tuple(new_pair)] += count
        vocab = new_vocab
  return merges

In [ ]:
corpus='''Gong Ji-hyeok and Go Da-rim work together to bring Go Da-rim’s ex-boyfriend into a business deal. But Ji-hyeok, a usually sharp and calm team leader, is caught off guard when he is kissed by Da-rim. Their relationship ends abruptly as Da-rim quickly departs back home as she hears news from her family. Go Da-rim, hiding the fact she is a single woman, pretends to be a mother to earn a living and pay her family’s debt and mother’s medical bills. On her first day she realises Gong Ji-hyeok is her boss. Da-rim needs to continue the pretence as they work together to pay the debt but the initial goals start to give way to unexpected feelings.'''

In [ ]:
x=learn_bpe(corpus,3)
print(x)

[('<', 'f'), ('e', 'e'), ('i', 'n')]
